# Preference Tree Optimization (PTO) on Colab

Trains a Llama-3.2-1B therapist policy with PTO and evaluates it with an LLM judge, using the
`pto` command-line tool from [preference-tree-optimization](https://github.com/Lior-Baruch/preference-tree-optimization).

**Before you start**

1. *Runtime → Change runtime type → GPU.* An A100 matches the reported runs; smaller GPUs work
   for the quick test.
2. Add these keys under *Secrets* (the key icon in the left sidebar) and enable notebook access:
   - `OPENAI_API_KEY`: patient simulator and oracle (required)
   - `HF_TOKEN`: Llama 3.2 is gated; accept its license on the Hugging Face Hub first (required)
   - `ANTHROPIC_API_KEY`: held-out judge (optional)
   - `WANDB_API_KEY`: Weights & Biases logging (optional)

Training calls the OpenAI API many times: a full 10-iteration run makes roughly 120k (K = 0) to
300k (K = 5) API calls. Start with the quick test.

## 1. Install

In [ ]:
# Colab already ships a CUDA build of PyTorch, so only the package and its extras are installed.
!git clone -q https://github.com/Lior-Baruch/preference-tree-optimization.git
%cd preference-tree-optimization
!pip install -q -e ".[train,eval,tracking]"
# peft rejects some torchao versions preinstalled on Colab; the project does not use torchao.
!pip uninstall -y -q torchao

## 2. API keys from Colab secrets

In [ ]:
import os

from google.colab import userdata

for name in ["OPENAI_API_KEY", "HF_TOKEN", "ANTHROPIC_API_KEY", "WANDB_API_KEY"]:
    try:
        value = userdata.get(name)
    except Exception:  # secret missing or notebook access not granted
        value = None
    if value:
        os.environ[name] = value
    print(f"{name:18s} {'set' if value else 'not set'}")

## 3. Where to write runs

Colab VMs are wiped when the session ends. Runs resume automatically, so writing them to Google
Drive turns a disconnect into a short pause instead of a restart.

In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/pto_runs"
else:
    OUTPUT_DIR = "/content/pto_runs"
print("Runs go to", OUTPUT_DIR)

## 4. Quick test

Two short iterations on four conversations: checks the GPU, the keys and the whole pipeline.

In [ ]:
!pto train --config configs/quicktest.yaml --output-dir {OUTPUT_DIR}

## 5. Full run

`configs/pto_k5.yaml` is the reported look-ahead run (K = 5); `configs/pto_k0.yaml` is the
no-look-ahead baseline. Expect many hours on an A100; re-run the cell after a disconnect to
resume.

In [ ]:
CONFIG = "configs/pto_k5.yaml"
!pto train --config {CONFIG} --dry-run --output-dir {OUTPUT_DIR}
!pto train --config {CONFIG} --output-dir {OUTPUT_DIR}

## 6. Evaluate

Score the conversations of the base model (iteration 0) and of the final policy (iteration 10)
with the training judge, then compare them persona by persona. Add
`--judge anthropic:claude-haiku-4-5` runs for the held-out judge.

In [ ]:
RUN = f"{OUTPUT_DIR}/pto_Q1Q2_Llama32-1B_K5_MCL12_M8_greedy"
JUDGE = "openai:gpt-4o-mini-2024-07-18"

!pto evaluate --conversations {RUN}/conversations/model_iter_0 --judge {JUDGE} --out {RUN}/scores_iter0.csv
!pto evaluate --conversations {RUN}/conversations/model_iter_10 --judge {JUDGE} --out {RUN}/scores_iter10.csv
!pto compare {RUN}/scores_iter0.csv {RUN}/scores_iter10.csv --out {RUN}/iter0_vs_iter10.md

## 7. Training curves

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUN}